<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediction - Trained Model Inference

This notebook uses a trained custom NER model to make predictions on new texts.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- Trained models are loaded from Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **Model Loading** - Load trained custom NER model from Google Drive
4. **Prediction Pipeline** - Create prediction pipeline
5. **Make Predictions** - Run predictions on new texts
6. **Visualize Results** - Display and save prediction results

**Requirements:**
- `training.ipynb` notebook must be run first
- Trained model must exist at `models/trained/custom_ner_model` in Google Drive


## 1. Google Drive Connection


In [2]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['predictions']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [3]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please upload spark_jsl.json to Google Drive at the project folder")
    print("You can upload it manually or use the following code:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [6]:
# Install Java (required for Spark)
import subprocess

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")

# Check GPU availability
# gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
# has_gpu = gpu_check.returncode == 0

# if has_gpu:
#     print("🚀 GPU detected! Installing PyTorch with CUDA support...")
#     %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# else:
#     print("Installing PyTorch (CPU version)...")
#     %pip install -q torch torchvision torchaudio

# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy

print("✅ All libraries installed successfully!")
# if has_gpu:
#     print("✅ GPU-accelerated PyTorch installed")


✅ Java is already installed: openjdk version "11.0.28" 2025-07-15
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.0/737.0 kB 32.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 10.1 MB/s eta 0:00:00
✅ All libraries installed successfully!


In [7]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql.types import StringType
from pyspark.sql import Row
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "8G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


⚠️  No GPU detected. Using CPU mode.
Starting Spark session...
✅ Spark NLP Version: 6.1.3
✅ Spark NLP JSL Version: 6.1.1
✅ Spark session initialized successfully


In [8]:
# Load trained model from Google Drive
model_path = f"{PROJECT_FOLDER}/models/trained/custom_ner_model"

if not os.path.exists(model_path):
    print(f"❌ Model not found: {model_path}")
    print("Please run training.ipynb first to train the model.")
    raise FileNotFoundError(f"Model not found at {model_path}")

print(f"Loading trained model from {model_path}...")
custom_model = MedicalNerModel.load(model_path)
print("✅ Model loaded successfully")

# Load embeddings (required for the model)
print("Loading embeddings...")
embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")
print("✅ Embeddings loaded")


Loading trained model from /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model...
✅ Model loaded successfully
Loading embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Embeddings loaded


## 4. Create Prediction Pipeline


In [9]:
# Create prediction pipeline
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")\
    .setCleanupMode("shrink")

sentence_detector = SentenceDetector()\
    .setInputCols(["document"])\
    .setOutputCol("sentence")\
    .setExplodeSentences(True)

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

# Configure model
custom_model.setInputCols(["sentence", "token", "embeddings"])\
    .setOutputCol("ner")

# NER converter to extract chunks
ner_converter = NerConverter()\
    .setInputCols(["document", "token", "ner"])\
    .setOutputCol("ner_chunks")

# Create pipeline
prediction_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    embeddings,
    custom_model,
    ner_converter
])

print("✅ Prediction pipeline created")


✅ Prediction pipeline created


## 5. Prepare Input Texts


In [12]:
from pyspark.sql import Row
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# ---------------------------
# 1. Daha zengin sample metinler
# ---------------------------
sample_texts = [
    "The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure closely due to hypertension.",
    "Dr. Smith recommended Metformin 500mg for type 2 diabetes mellitus and suggested regular HbA1c testing.",
    "The patient has a history of hypertension, hyperlipidemia, and is currently taking Lisinopril 10mg once daily, along with Atorvastatin 20mg at night.",
    "Patient presents with chest pain, shortness of breath, and palpitations. ECG shows ST elevation, and troponin levels are elevated.",
    "The medication dosage was increased to 20mg per day after consultation. The patient also started Omeprazole 40mg daily for gastroesophageal reflux disease.",
    "Administered Vancomycin 1g IV every 12 hours. Monitor renal function and complete blood count during therapy."
]

# ---------------------------
# 2. Create Spark DataFrame
# ---------------------------
text_df = spark.createDataFrame([Row(text=text) for text in sample_texts])
print(f"✅ Created DataFrame with {text_df.count()} texts")
text_df.show(truncate=150)

# ---------------------------
# 3. Fit pipeline and transform
# ---------------------------
print("Running prediction pipeline...")
pipeline_model = prediction_pipeline.fit(text_df)
predictions = pipeline_model.transform(text_df)
print("✅ Predictions completed")



✅ Created DataFrame with 6 texts
+------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                                  text|
+------------------------------------------------------------------------------------------------------------------------------------------------------+
|       The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure closely due to hypertension.|
|                                               Dr. Smith recommended Metformin 500mg for type 2 diabetes mellitus and suggested regular HbA1c testing.|
| The patient has a history of hypertension, hyperlipidemia, and is currently taking Lisinopril 10mg once daily, along with Atorvastatin 20mg at night.|
|                    Patient presents with chest 

## 6. Run Predictions


In [13]:

# ---------------------------
# 4. UDF
# ---------------------------
def highlight_entities(text, chunks):
    """NER çıktısını renklendirir: her entity türü farklı renk"""
    if not chunks:
        return text
    colors = {
        "Drug": "\033[91m",       # red
        "Dosage": "\033[92m",     # green
        "Frequency": "\033[94m",  # blur
        "Route": "\033[95m",      # pink
        "Duration": "\033[96m",   # cam göbeği
        "Symptom": "\033[93m",    # yellow
        "Test": "\033[90m",       # gri
        "Disease": "\033[31m",    # koyu kırmızı
        "PHI": "\033[35m"         # pembe
    }
    highlighted = text
    for chunk in chunks:
        entity = chunk.result
        label = chunk.metadata.get("entity", "UNK")
        color = colors.get(label, "\033[0m")
        highlighted = highlighted.replace(entity, f"{color}{entity}\033[0m")
    return highlighted

highlight_udf = udf(highlight_entities, StringType())

# ---------------------------
# 5. Apply UDF and show
# ---------------------------
predictions_with_highlight = predictions.withColumn("highlighted_text", highlight_udf("text", "ner_chunks"))
predictions_with_highlight.select("highlighted_text", "ner_chunks").show(truncate=200)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                             highlighted_text|                                                                                                                  ner_chunks|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------+
|              The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure closely due to 